In [ ]:
# 데이터 로드
from torchvision import datasets, transforms
import torch

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root= "./data",
    train = True,
    download= True,
    transform=transform
)

# train/valid 분리
from torch.utils.data import random_split

train_dataset, valid_dataset = random_split(
    train_dataset, [48000, 12000]
)

# 데이터 셋/데이터 로더 만들기
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size= 64, shuffle=True) # 데이터를 한번에 64개씩 잘라서 모델에 넣어주는 역할 / 전체데이터를 한번에 넣으면 너무 느리고 메모리가 터지기 때문
valid_loader = DataLoader(valid_dataset, batch_size= 64, shuffle=False) # 배치사이즈 = 데이터를 몇개씩 묶어서 넣을건지 / 셔플은 데이터를 섞어 넣을건지 유무

# 모델 만들기
import torch.nn as nn

class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential( # 특징 추출 (Conv 영역)
            nn.Conv2d(1, 32, kernel_size=3, padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32, kernel_size=3, padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #28 -> 14

            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #14 -> 7

            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 7 -> 3

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.classifier = nn.Sequential( # 분류 (Linear 영역)
            nn.Flatten(), # Linear가 원하는 입력 형태는 2차원 입력(batch_size, feature 수) 예 (64개의 데이터, 1152개의 특징) 인데 flatten(=x.view(x.size(0),-1))같은 평면화 하지않으면 (64, 128, 3, 3) 이렇게 생김
                          # flatten은 모양을 바꾸는 작업 이 밑의 Linear는 데이터를 학습하는 작업
            nn.Linear(128 * 3 * 3, 256), # 1152개의 특징 -> 256개의 중요한 특징으로 압축
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
        
# 모델/손실함수/옵티마이저 준비
model = BetterCNN() # 모델 생성

criterion = nn.CrossEntropyLoss() # Cross = 교차 / Entropy = 정보량(불확실성) / Loss = 손실(틀린 정도) / 예측과 정답이 얼마나 다른지 계산하는 함수
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001) # optimizer = 최적화 도구 / 틀린만큼 모델을 어떻게 고칠지 결정
                                                            # Adam = Adaptive Moment Estimation 똑똑하게 가중치를 수정해주는 알고리즘
                                                            # model.parameters() = 모델 안의 모든 가중치(weight) Ex) Conv, Linear
                                                            # lr = Learning rate / 얼마나 크게 수정시킬지 / 너무 크면 왔다갔다 폭발, 너무작으면 너무느려서 거의안배움

# 학습 코드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # GPU or CPU Mac은 NVIDIA CUDA를 지원안해서 CPU
model = model.to(device) # 모델을 CPU에 적용시키는 작업 / 데이터와 모델은 같은 장치에 있어야함

epochs = 10 # 반복 횟수 / 훈련 데이터 전체를 한바퀴 다보는게 1 epoch
best_acc = 0 # 최고 좋은 epoch를 찾는 변수
patience = 2
counter = 0

for epoch in range(epochs): # epoch 횟수만큼 학습 반복
    model.train() # 모델을 학습모드로 전환 training
    train_loss = 0 # epoch동안 손실을 누적해서 기록하기 위한 변수

    for images, labels in train_loader: # train 데이터를 batch단위로 하나씩 꺼내오기
        images, labels = images.to(device), labels.to(device) # 데이터들도 model과 같은 디바이스로 옮겨주기

        optimizer.zero_grad() # 이전 batch에서 계산된 기울기를 초기화 / Pytorch는 기울기를 계속 누적하는데 초기화를 안하면 이전것과 이번것이 섞여버림 / 이번 batch 기준으로 업데이트
        outputs = model(images) # 모델이 이미지를 보고 예측값 생성 / ex) (64,10).. 64장의 이미지 각 이미지마다 10개의 숫자에 대한 점수가 10개
        loss = criterion(outputs, labels) # 모델 예측이 정답과 얼마나 다른지 계산 / 예를들어 정답이 2인데 모델이 2에 높은점수를 주면 loss가 낮고, 다른수에 높은점수를 주면 loss가 높다.
        loss.backward() # 위에서 계산한 오차를 바탕으로 어떤 가중치를 얼마나 고쳐야 하는지 계산하는 작업 / 틀린 원인을 뒤로 거슬러 올라가며 각 가중치의 책임을 계산 (정답 비교-> 오차 계산-> 뒤로 전달)
        optimizer.step() # 방금 계산한 기울기(gradient)를 이용해 실제로 가중치를 수정 / loss.backward() = 오답 분석, optimizer.step() = 실제 수정

        train_loss += loss.item() # 현재 batch의 loss를 숫자로 꺼내서 변수에 누적 / loss.item()은 tensor형태(파이썬숫자)라서 출력용 숫자로 변형하기위해 .item()을 사용
    avg_train_loss = train_loss / len(train_loader)

    model.eval() # 모델을 평가모드로 전환
    correct = 0 # 검증 데이터에서 맞춘 개수 카운트하려는 변수
    total = 0 # 검증 데이터의 전체 개수 카운트하는 변수

    with torch.no_grad(): # 검증할 때에는 gradient 계산하지 말라는 명령어 / 검증할때에는 학습 안하고 평가만 하니 역전파용 계산이 무쓸모.
        for images, labels in valid_loader: # 검증 데이터도 batch 단위로 꺼내오기 단, 이곳에선 train처럼 학습하지 않고 평가만 함.
            images, labels = images.to(device), labels.to(device)
            outputs = model(images) # 검증 데이터에 대해 모델의 예측
            _, predicted = torch.max(outputs, 1) # torch.max(outputs, 1)은 가장 큰값, 그 위치(index)를 반환하는데 우리는 index만 필요하니 앞의 값은 버려줌.
                                                 # 1의 의미는 Pytorch에서 0은 세로방향(행) 1이 가로방향(열)을 의미
                                                 # 코드 의미: 각 행(= 각 이미지)에 대해 가장 큰 값과 그 위치를 찾아라 / 각 행에서 가로방향으로 최대값 찾기 > 열방향으로 움직이며 계산
                                                 # dim = 그 축을 없애면서 연산, dim=1 > 열 방향으로 계산 > 행마다 결과
                                                 # dim=0 > 행 방향으로 계산 > 열마다 결과 / dim은 어느방향으로 훑을지!

            total += labels.size(0) # 이번 batch의 데이터 개수를 total에 더해주기. batch size가 64면 한번에 64추가
            correct += (predicted == labels).sum().item() # 이번 batch에서 맞춘 개수 세기 >> 예측과 정답을 비교해서 .sum()으로 True를 1처럼 세서 계산 후 .item()으로 숫자로 바꿔줌

        accuracy = correct / total # 전체 중에서 맞춘 비율

        if accuracy > best_acc: # best_acc변수에 계속 최고 accuracy를 넣어줌
            best_acc = accuracy 
            torch.save(model.state_dict(), "best_model.pth") # 현재 모델의 학습된 가중치를 파일로 저장, 모델에 학습된 모든 값을 파이썬 객체 파일로 저장하는 함수, .pth는 파일 확장자+저장할 파일이름
                                                             # state_dict는 모델이 학습한 모든 지식(뇌)라고 생각하면 됨. 
            counter = 0 # 카운터 초기화
        else:
            counter += 1 # 최고 accuracy보다 안좋은 accuracy가 나온 횟수를 카운트
        
        if counter >= patience:
            print("Early Stopping!")
            break
        
        # 추후에 파일 불러와서 사용할 때
        # model = BetterCNN()  구조 먼저 만들고
        # model.load_state_dict(torch.load("best_model.pth")) 파일 불러오기
        # model.eval() 시험 모드 시작
        
        print(f"Epoch {epoch+1}/{epochs}, Train loss: {avg_train_loss:.4f}, Valid Accuracy: {accuracy:.4f}") # 현재 epoch의 학습상태 출력

In [ ]:
# augmentation 적용 버전

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torch
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

train_transform = transforms.Compose([
    transforms.RandomRotation(8), # 회전 10도 랜덤
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)), # 위치 살짝 이동
    transforms.ToTensor()
])

valid_transform = transforms.Compose([
    transforms.ToTensor()
])

full_train_aug = datasets.FashionMNIST( # 훈련용 데이터를 변형하기 위한 데이터와 valid용 원본 데이터 두개 로드
    root = "./data",
    train=True,
    download=True, # 데이터가 없으면 자동 다운로드 하라는 코드.
    transform=train_transform
)

full_train_plain = datasets.FashionMNIST(
    root= "./data",
    train=True,
    download=True,
    transform=valid_transform
)

train_size = 48000 # 4:1비율 훈련, 검증 데이터 나누기
valid_size = 12000

g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_train_aug), generator=g).tolist() # randperm: 데이터 인덱스를 랜덤하게 섞는 도구 / full_train_aug는 60000개의 전체데이터를 넣기위해 / tolist는 텐서데이터를 파이썬 리스트로 변환해주는것
                                                       # 리스트로 변환해주는 이유는 Subset이 list를 더 편하게 받기 때문이다. (관례적으로 사용함)
train_indices = indices[:train_size] # 앞 48000개 train
valid_indices = indices[train_size:] # 뒤 12000개 valid / 데이터를 랜덤하게 train, valid 분리

train_dataset = Subset(full_train_aug, train_indices) # Subset(데이터셋, 인덱스리스트) : 전체 데이터 중에서 특정 번호(인덱스)들만 골라서 새로운 데이터셋을 생성해라.
valid_dataset = Subset(full_train_plain, valid_indices) # augmentation이 적용된 데이터에서 train_indices에 해당되는 것만 사용해라. valid도 마찬가지(plain이 적용된 데이터에서 valid_indices에 해당하는 것만 사용)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle= True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle= False)

import torch.nn as nn

class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential( # 특징 추출 (Conv 영역)
            nn.Conv2d(1, 32, kernel_size=3, padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32, kernel_size=3, padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #28 -> 14

            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #14 -> 7

            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 7 -> 3
        )

        self.classifier = nn.Sequential( # 분류 (Linear 영역)
            nn.Flatten(), # Linear가 원하는 입력 형태는 2차원 입력(batch_size, feature 수) 예 (64개의 데이터, 1152개의 특징) 인데 flatten(=x.view(x.size(0),-1))같은 평면화 하지않으면 (64, 128, 3, 3) 이렇게 생김
                          # flatten은 모양을 바꾸는 작업 이 밑의 Linear는 데이터를 학습하는 작업
            nn.Linear(128 * 3 * 3, 256), # 1152개의 특징 -> 256개의 중요한 특징으로 압축
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
        
# 모델/손실함수/옵티마이저 준비
model = BetterCNN() # 모델 생성

criterion = nn.CrossEntropyLoss() # Cross = 교차 / Entropy = 정보량(불확실성) / Loss = 손실(틀린 정도) / 예측과 정답이 얼마나 다른지 계산하는 함수
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001) # optimizer = 최적화 도구 / 틀린만큼 모델을 어떻게 고칠지 결정
                                                            # Adam = Adaptive Moment Estimation 똑똑하게 가중치를 수정해주는 알고리즘
                                                            # model.parameters() = 모델 안의 모든 가중치(weight) Ex) Conv, Linear
                                                            # lr = Learning rate / 얼마나 크게 수정시킬지 / 너무 크면 왔다갔다 폭발, 너무작으면 너무느려서 거의안배움

# 학습 코드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # GPU or CPU Mac은 NVIDIA CUDA를 지원안해서 CPU
model = model.to(device) # 모델을 CPU에 적용시키는 작업 / 데이터와 모델은 같은 장치에 있어야함

epochs = 20 # 반복 횟수 / 훈련 데이터 전체를 한바퀴 다보는게 1 epoch
best_acc = 0 # 최고 좋은 epoch를 찾는 변수
patience = 3
counter = 0

for epoch in range(epochs): # epoch 횟수만큼 학습 반복
    model.train() # 모델을 학습모드로 전환 training
    train_loss = 0 # epoch동안 손실을 누적해서 기록하기 위한 변수

    for images, labels in train_loader: # train 데이터를 batch단위로 하나씩 꺼내오기
        images, labels = images.to(device), labels.to(device) # 데이터들도 model과 같은 디바이스로 옮겨주기

        optimizer.zero_grad() # 이전 batch에서 계산된 기울기를 초기화 / Pytorch는 기울기를 계속 누적하는데 초기화를 안하면 이전것과 이번것이 섞여버림 / 이번 batch 기준으로 업데이트
        outputs = model(images) # 모델이 이미지를 보고 예측값 생성 / ex) (64,10).. 64장의 이미지 각 이미지마다 10개의 숫자에 대한 점수가 10개
        loss = criterion(outputs, labels) # 모델 예측이 정답과 얼마나 다른지 계산 / 예를들어 정답이 2인데 모델이 2에 높은점수를 주면 loss가 낮고, 다른수에 높은점수를 주면 loss가 높다.
        loss.backward() # 위에서 계산한 오차를 바탕으로 어떤 가중치를 얼마나 고쳐야 하는지 계산하는 작업 / 틀린 원인을 뒤로 거슬러 올라가며 각 가중치의 책임을 계산 (정답 비교-> 오차 계산-> 뒤로 전달)
        optimizer.step() # 방금 계산한 기울기(gradient)를 이용해 실제로 가중치를 수정 / loss.backward() = 오답 분석, optimizer.step() = 실제 수정

        train_loss += loss.item() # 현재 batch의 loss를 숫자로 꺼내서 변수에 누적 / loss.item()은 tensor형태(파이썬숫자)라서 출력용 숫자로 변형하기위해 .item()을 사용
    avg_train_loss = train_loss / len(train_loader)

    model.eval() # 모델을 평가모드로 전환
    correct = 0 # 검증 데이터에서 맞춘 개수 카운트하려는 변수
    total = 0 # 검증 데이터의 전체 개수 카운트하는 변수

    with torch.no_grad(): # 검증할 때에는 gradient 계산하지 말라는 명령어 / 검증할때에는 학습 안하고 평가만 하니 역전파용 계산이 무쓸모.
        for images, labels in valid_loader: # 검증 데이터도 batch 단위로 꺼내오기 단, 이곳에선 train처럼 학습하지 않고 평가만 함.
            images, labels = images.to(device), labels.to(device)
            outputs = model(images) # 검증 데이터에 대해 모델의 예측
            _, predicted = torch.max(outputs, 1) # torch.max(outputs, 1)은 가장 큰값, 그 위치(index)를 반환하는데 우리는 index만 필요하니 앞의 값은 버려줌.
                                                 # 1의 의미는 Pytorch에서 0은 세로방향(행) 1이 가로방향(열)을 의미
                                                 # 코드 의미: 각 행(= 각 이미지)에 대해 가장 큰 값과 그 위치를 찾아라 / 각 행에서 가로방향으로 최대값 찾기 > 열방향으로 움직이며 계산
                                                 # dim = 그 축을 없애면서 연산, dim=1 > 열 방향으로 계산 > 행마다 결과
                                                 # dim=0 > 행 방향으로 계산 > 열마다 결과 / dim은 어느방향으로 훑을지!

            total += labels.size(0) # 이번 batch의 데이터 개수를 total에 더해주기. batch size가 64면 한번에 64추가
            correct += (predicted == labels).sum().item() # 이번 batch에서 맞춘 개수 세기 >> 예측과 정답을 비교해서 .sum()으로 True를 1처럼 세서 계산 후 .item()으로 숫자로 바꿔줌

        accuracy = correct / total # 전체 중에서 맞춘 비율

        if accuracy > best_acc: # best_acc변수에 계속 최고 accuracy를 넣어줌
            best_acc = accuracy 
            torch.save(model.state_dict(), "best_model.pth") # 현재 모델의 학습된 가중치를 파일로 저장, 모델에 학습된 모든 값을 파이썬 객체 파일로 저장하는 함수, .pth는 파일 확장자+저장할 파일이름
                                                             # state_dict는 모델이 학습한 모든 지식(뇌)라고 생각하면 됨. 
            counter = 0 # 카운터 초기화
        else:
            counter += 1 # 최고 accuracy보다 안좋은 accuracy가 나온 횟수를 카운트
        
        if counter >= patience:
            print("Early Stopping!")
            break
        
        # 추후에 파일 불러와서 사용할 때
        # model = BetterCNN()  구조 먼저 만들고
        # model.load_state_dict(torch.load("best_model.pth")) 파일 불러오기
        # model.eval() 시험 모드 시작
        
        print(f"Epoch {epoch+1}/{epochs}, Train loss: {avg_train_loss:.4f}, Valid Accuracy: {accuracy:.4f}") # 현재 epoch의 학습상태 출력